In [0]:
import dlt

In [0]:
#create a streaming table for orders
@dlt.table(
    table_properties = {"quality":"bronze"},
    comment = "Orders_bronze table"
)

def orders_bronze():
    df = spark.readStream.table("dev.bronze.orders_raw")
    return df

In [0]:
#create a meterlized view for customer
@dlt.table(
    table_properties = {"quality":"bronze"},
    comment = "Customer_bronze table",
    name = "customer_bronze"
)

def cust_bronze():
    df = spark.read.table("dev.bronze.customer_raw")
    return df

In [0]:
#create a view to join orders and customer
@dlt.view(
    comment = "joined view",
)

def joined_view():
    df_c = spark.read.table("LIVE.customer_bronze")
    df_o = spark.read.table("LIVE.orders_bronze")
    
    df_join = df_o.join(df_c, how="left_outer", on=df_c.c_custkey == df_o.o_custkey)
    
    return df_join


In [0]:
from pyspark.sql.functions import *
#create a meterlized view for customer
@dlt.table(
    table_properties = {"quality":"silver"},
    comment = "joined silver table",
    name = "joined_silver"
)

def joined_silver():
    df = spark.read.table("LIVE.joined_view").withColumn("insert_date",current_timestamp())
    return df

In [0]:
from pyspark.sql.functions import *
#create a meterlized view for customer
@dlt.table(
    table_properties = {"quality":"gold"},
    comment = "orders aggregted gold"
)

def orders_agg_gold():
    df = spark.read.table("LIVE.joined_silver")
    df_final = df.groupby("c_mktsegment") \
                .agg(count("o_orderkey").alias("sum_orders")) \
                .withColumn("_insert_date", current_timestamp())
    return df_final


In [0]:
%sql
select * from dev.etl.orders_agg_gold